7. For fixed ε, compute the average solution times as a function of n. Compare the obtained solution times with the SAA (ε = 0) and the robust optimization approach (ε is sufficiently large).

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

def solve_with_epsilon(env, generator, n, epsilon, K_train=30):

    # Решает задачу для заданного n и epsilon, возвращает время решения.

    samples = generator.generate_samples(n, K_train)
    D, g = build_box_uncertainty(samples)
    
    solver = DROAssignment(
        n=n,
        epsilon=epsilon,
        alpha=0.95,
        D=D,
        g=g,
        env=env
    )
    
    start = time.time()
    x_opt, obj = solver.solve(samples)
    elapsed = time.time() - start
    
    return elapsed

def run_task_7(env, generator, n_values, eps_fixed=0.1, eps_large=1e4, num_runs=5):
    # Для каждого n несколько раз решаем задачу при разных ε, усредняем время.
    times_saa = []
    times_dro = []
    times_robust = []
    
    for n in n_values:
        print(f"Processing n = {n}...")
        
        t_saa = []
        t_dro = []
        t_robust = []
        
        for _ in range(num_runs):
            t_saa.append(solve_with_epsilon(env, generator, n, epsilon=0.0))
            t_dro.append(solve_with_epsilon(env, generator, n, epsilon=eps_fixed))
            t_robust.append(solve_with_epsilon(env, generator, n, epsilon=eps_large))
        
        times_saa.append(np.mean(t_saa))
        times_dro.append(np.mean(t_dro))
        times_robust.append(np.mean(t_robust))
    
    return times_saa, times_dro, times_robust

# Параметры эксперимента
n_values_task7 = [5, 10, 15, 20, 25]
num_runs = 3

env = gp.Env(empty=True)
set_environ(env)
env.start()

generator = DRODataGenerator(seed=42)

times_saa, times_dro, times_robust = run_task_7(
    env, generator, n_values_task7, 
    eps_fixed=0.1, eps_large=1e4, 
    num_runs=num_runs
)

env.close()

# Построение графиков
plt.figure(figsize=(12, 5))

# График 1: Зависимость времени от n для фиксированного ε
plt.subplot(1, 2, 1)
plt.plot(n_values_task7, times_dro, marker='o', label=f'DRO (ε=0.1)')
plt.plot(n_values_task7, times_saa, marker='s', label='SAA (ε=0)')
plt.plot(n_values_task7, times_robust, marker='^', label='Robust (ε large)')
plt.xlabel('n (размер задачи)')
plt.ylabel('Среднее время решения (сек)')
plt.title('Зависимость времени решения от n')
plt.legend()
plt.grid(True)

# График 2: Сравнение времени для разных ε при фиксированном n (например, n=15)
idx_n = n_values_task7.index(15)
labels = ['SAA (ε=0)', 'DRO (ε=0.1)', 'Robust (ε large)']
times_at_n15 = [times_saa[idx_n], times_dro[idx_n], times_robust[idx_n]]

plt.subplot(1, 2, 2)
plt.bar(labels, times_at_n15, color=['blue', 'green', 'red'])
plt.ylabel('Время решения (сек)')
plt.title(f'Сравнение подходов при n=15')
plt.grid(axis='y')

plt.tight_layout()
plt.show()

# Вывод численных значений
print("\n=== Среднее время решения (сек) ===")
print("n\tSAA (ε=0)\tDRO (ε=0.1)\tRobust (ε large)")
for i, n in enumerate(n_values_task7):
    print(f"{n}\t{times_saa[i]:.3f}\t\t{times_dro[i]:.3f}\t\t{times_robust[i]:.3f}")